In [1]:
import pandas as pd
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import(
  accuracy_score,
  precision_score,
  recall_score,
  f1_score,
  roc_auc_score,
  confusion_matrix,
  classification_report
)

In [2]:
current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

split_path = project_root / "data" / "processed" / "splits"

X_train = pd.read_csv(split_path / "X_train.csv")
X_test = pd.read_csv(split_path / "X_test.csv")
y_train = pd.read_csv(split_path / "y_train.csv").iloc[:, 0]
y_test = pd.read_csv(split_path / "y_test.csv").iloc[:, 0]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)



X_train: (2904, 36)
X_test: (726, 36)
y_train: (2904,)
y_test: (726,)


In [3]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

print("XGBoost baseline model trained successfully.")

XGBoost baseline model trained successfully.


In [4]:
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print("Predictions completed.")

Predictions completed.


In [5]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

metrics = pd.DataFrame({
    "Model": ["XGBoost Baseline"],
    "Accuracy": [accuracy],
    "Precision": [precision],
    "Recall": [recall],
    "F1-score": [f1],
    "ROC-AUC": [roc_auc]
})

metrics

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,XGBoost Baseline,0.929752,0.92674,0.890845,0.908438,0.974046


In [6]:
print("Classification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.95      0.94       442
           1       0.93      0.89      0.91       284

    accuracy                           0.93       726
   macro avg       0.93      0.92      0.93       726
weighted avg       0.93      0.93      0.93       726

Confusion Matrix:
[[422  20]
 [ 31 253]]


In [7]:
results_path = project_root / "results"
models_path = project_root / "models"

results_path.mkdir(parents=True, exist_ok=True)
models_path.mkdir(parents=True, exist_ok=True)

metrics.to_csv(results_path / "xgboost_baseline_metrics.csv", index=False)

xgb_model.save_model(models_path / "xgboost_baseline_model.json")

print("Metrics saved to:", results_path / "xgboost_baseline_metrics.csv")
print("Model saved to:", models_path / "xgboost_baseline_model.json")

Metrics saved to: c:\Users\sandy\Documents\git\student-dropout-xai-pipeline\results\xgboost_baseline_metrics.csv
Model saved to: c:\Users\sandy\Documents\git\student-dropout-xai-pipeline\models\xgboost_baseline_model.json
